# Week 1 — Task 5: U-Net smoke test (20 images, 2 epochs)

Covers tracker sub-tasks 5.1–5.8 (parent task "5. U-Net smoke test (20 images, 2 epochs)"):

| Task ID | Sub-task |
|---|---|
| 5.1 | Set up the environment |
| 5.2 | Write/hand-roll the U-Net architecture |
| 5.3 | Wire dataset.py to a 20-image debug subset |
| 5.4 | Set up loss and optimizer |
| 5.5 | Run the training loop for 2 epochs |
| 5.6 | Confirm the loss actually decreases |
| 5.7 | Save a checkpoint and reload it |
| 5.8 | Log runtime, GPU memory, and any bugs hit |

**Goal (per the tracker): prove the U-Net plumbing works end-to-end — this is not a real
training run.** 20 images, 2 epochs, no augmentation, no validation split. Real training
(augmentation, early stopping, checkpointing-by-config, all 4 brackets × 3 seeds) is Week 2/3.

This notebook is best run on Colab (per the project's compute plan) since `torch.cuda` is
what task 5.1 explicitly asks you to confirm — the code below also falls back to Apple
Silicon `mps` or CPU so it still runs locally if you're just sanity-checking it first.

Assumes this notebook lives in `notebooks/` (same convention as the Week 1 Task 2 notebook)
and the dataset is unzipped under `data/kvasir-seg/`, with the frozen splits already
committed under `data/splits/` (they are — `train_10.txt` etc. already exist in this repo).

**The U-Net model itself lives in `src/unet.py`** (Task 5.2), not in this notebook — it's a
shared building block, same reasoning as `dataset.py`/`metrics.py`/`evaluate.py` living in
`src/` rather than being copy-pasted into every notebook that needs it.

In [1]:
from pathlib import Path
import sys
import time
from datetime import date

import numpy as np
import torch
import torch.nn as nn
import yaml
from torch.utils.data import DataLoader

REPO_ROOT = Path.cwd().parent  # assumes notebooks/
sys.path.insert(0, str(REPO_ROOT / "src"))

from dataset import KvasirSegDataset  # noqa: E402
from unet import UNet  # noqa: E402

DATA_ROOT_CANDIDATES = [REPO_ROOT / "data" / "kvasir-seg", REPO_ROOT / "data" / "kvasir-seg" / "Kvasir-SEG"]
DATA_ROOT = next((c for c in DATA_ROOT_CANDIDATES if (c / "images").is_dir() and (c / "masks").is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"Couldn't find images/ and masks/ under any of {DATA_ROOT_CANDIDATES}.")

SPLITS_DIR = REPO_ROOT / "data" / "splits"
DECISIONS_MD = REPO_ROOT / "decisions.md"
CHECKPOINTS_DIR = REPO_ROOT / "checkpoints"  # already in .gitignore — safe to write .pt files here
CHECKPOINTS_DIR.mkdir(exist_ok=True)

with open(REPO_ROOT / "configs" / "unet_base.yaml") as f:
    UNET_CFG = yaml.safe_load(f)

print(f"Data:   {DATA_ROOT}")
print(f"Config: model={UNET_CFG['model']}, input={UNET_CFG['input']}, "
      f"training.loss={UNET_CFG['training']['loss']}, "
      f"training.optimizer={UNET_CFG['training']['optimizer']}, training.lr={UNET_CFG['training']['lr']}")

results_log = {}  # task_id -> summary string, filled in by each cell below

Data:   /Volumes/Crucial X9/home/Developer/COMPSCI-764-research/data/kvasir-seg
Config: model={'name': 'unet', 'depth': 4, 'base_channels': 32, 'pretrained': False}, input={'resize': [256, 256]}, training.loss=dice_bce, training.optimizer=adam, training.lr=0.001


In [3]:
print(torch.backends.mps.is_available())

True


## Task 5.1 — Set up the environment

Picks the best available device (`cuda` > `mps` > `cpu`) and reports the PyTorch version.

**Note on "confirm the PyTorch version matches what's pinned":** `requirements.txt`'s own
header says versions are *not yet pinned* — that's a standing TODO in the file itself, not
something this notebook can satisfy. Flagging it as WARN rather than pretending it passes.

In [2]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    device_name = torch.cuda.get_device_name(0)
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    device_name = "Apple Silicon (mps)"
else:
    DEVICE = torch.device("cpu")
    device_name = "CPU"

print(f"torch version: {torch.__version__}")
print(f"Selected device: {DEVICE} ({device_name})")
print(f"cuda.is_available(): {torch.cuda.is_available()}")

pinned_versions_exist = False  # requirements.txt header: "NOT yet pinned to exact versions (TODO)"

if DEVICE.type == "cuda":
    print("PASS: CUDA available.")
    results_log["5.1"] = f"PASS — CUDA available ({device_name}), torch {torch.__version__}."
else:
    print(f"[WARN] No CUDA device — running on {DEVICE}. Fine for this smoke test; "
          f"the real Week 3 grid (12 runs) should run on CUDA (Colab) per the compute plan.")
    results_log["5.1"] = (
        f"WARN — no CUDA device, ran on {DEVICE} ({device_name}), torch {torch.__version__}. "
        f"requirements.txt versions are not yet pinned (open TODO in the file itself)."
    )

torch version: 2.11.0
Selected device: mps (Apple Silicon (mps))
cuda.is_available(): False
[WARN] No CUDA device — running on mps. Fine for this smoke test; the real Week 3 grid (12 runs) should run on CUDA (Colab) per the compute plan.


## Task 5.2 — Write/hand-roll the U-Net architecture

Implemented in `src/unet.py` (4 down-blocks + 4 up-blocks, skip connections, base width 32
channels, no pretrained weights — matches `configs/unet_base.yaml`). This cell just
instantiates it from that config and checks the output shape matches the mask shape.

In [4]:
model = UNet(
    in_channels=3,
    out_channels=1,
    base_channels=UNET_CFG["model"]["base_channels"],
    depth=UNET_CFG["model"]["depth"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameters: {n_params:,}")

resize_h, resize_w = UNET_CFG["input"]["resize"]
dummy = torch.randn(2, 3, resize_h, resize_w, device=DEVICE)
with torch.no_grad():
    dummy_out = model(dummy)
print(f"Dummy input shape:  {tuple(dummy.shape)}")
print(f"Dummy output shape: {tuple(dummy_out.shape)}")

expected_shape = (2, 1, resize_h, resize_w)
if tuple(dummy_out.shape) == expected_shape:
    print("PASS: output shape matches mask shape.")
    results_log["5.2"] = (
        f"PASS — UNet(depth={UNET_CFG['model']['depth']}, base_channels={UNET_CFG['model']['base_channels']}) "
        f"in src/unet.py, {n_params:,} params, output shape {dummy_out.shape} matches mask shape, no pretrained weights."
    )
else:
    raise AssertionError(f"Output shape {tuple(dummy_out.shape)} != expected {expected_shape}")

UNet(
  (in_conv): DoubleConv(
    (block): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down1): Down(
    (block): Sequential(
      (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): DoubleConv(
        (block): Sequential(
          (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (4): Batch

## Task 5.3 — Wire dataset.py to a 20-image debug subset

`KvasirSegDataset` (in `src/dataset.py`) is jointly owned — this cell doesn't modify it, it
just constructs one against a throwaway 20-image debug split, exactly like the SAM 3/U-Net
arms already do against the real frozen splits.

Two things worth knowing about `dataset.py`'s actual contract before this will work:

1. When `resize` is set but no `transform` is given, it hands back a resized **PIL Image**
   and a **numpy** mask — not tensors. Training needs tensors, so this cell defines a small
   `transform` callable matching the contract `dataset.py`'s docstring describes
   (`image=np.ndarray, mask=np.ndarray -> {"image": ..., "mask": ...}`). No augmentation here
   on purpose — Task 5 is a plumbing test; the real augmentation pipeline is Week 2 Task 1.1.
2. **`dataset.py.__getitem__` returns a `KvasirSample` dataclass, not a dict/tuple — and
   PyTorch's default `DataLoader` collate function cannot batch a plain dataclass.** This
   would crash the very first `next(iter(loader))` with `batch_size > 1`. It's a real gap
   between the shared `dataset.py` and standard `DataLoader` usage that evaluate.py's
   per-image loop never exercises (and SAM 3's `configs/sam3_pvs.yaml` pins
   `inference.batch_size: 1`, so it wouldn't hit this either) — worth a heads-up to Sebastian
   since any future batched-training code on the U-Net side will hit the same thing. Fixed
   here with a small `collate_fn`, entirely inside this notebook — `dataset.py` itself is
   untouched.

In [5]:
def to_tensor_transform(image: np.ndarray, mask: np.ndarray) -> dict:
    """Matches dataset.py's transform contract. No augmentation — plumbing test only."""
    image_t = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0  # HWC uint8 -> CHW float32 [0,1]
    mask_t = torch.from_numpy(mask).float()  # HW {0,1} uint8 -> HW float32
    return {"image": image_t, "mask": mask_t}


def collate_kvasir_samples(batch: list) -> dict:
    """Default collate can't batch a KvasirSample dataclass — unpack it explicitly
    instead of changing the shared dataset.py."""
    images = torch.stack([s.image for s in batch], dim=0)
    masks = torch.stack([s.mask for s in batch], dim=0)
    image_ids = [s.image_id for s in batch]
    return {"image": images, "mask": masks, "image_id": image_ids}


N_DEBUG = 20
source_split = SPLITS_DIR / "train_10.txt"  # 70 images — plenty to take 20 from
debug_split_path = SPLITS_DIR / "debug_20.txt"

with open(source_split) as f:
    all_ids = [line.strip() for line in f if line.strip() and not line.startswith("#")]
debug_ids = all_ids[:N_DEBUG]

with open(debug_split_path, "w") as f:
    f.write("# Week 1 Task 5 smoke-test split ONLY — not one of the official nested\n")
    f.write("# brackets in configs/data.yaml. Do not use for real training runs.\n")
    for image_id in debug_ids:
        f.write(image_id + "\n")

print(f"Wrote {len(debug_ids)} debug IDs to {debug_split_path.relative_to(REPO_ROOT)}")

debug_dataset = KvasirSegDataset(
    data_root=str(DATA_ROOT),
    split_file=str(debug_split_path),
    resize=tuple(UNET_CFG["input"]["resize"]),
    transform=to_tensor_transform,
)
debug_loader = DataLoader(
    debug_dataset, batch_size=4, shuffle=True, collate_fn=collate_kvasir_samples
)

batch = next(iter(debug_loader))
print(f"Batch image shape: {tuple(batch['image'].shape)}  dtype={batch['image'].dtype}")
print(f"Batch mask shape:  {tuple(batch['mask'].shape)}  dtype={batch['mask'].dtype}")
print(f"Batch image_ids:   {batch['image_id']}")

assert batch["image"].shape[0] == 4 and batch["image"].shape[1:] == (3, *UNET_CFG["input"]["resize"])
assert batch["mask"].shape[0] == 4 and batch["mask"].shape[1:] == tuple(UNET_CFG["input"]["resize"])
print("PASS: DataLoader returns correctly-shaped batches, no errors.")
results_log["5.3"] = (
    f"PASS — {len(debug_dataset)}-image debug split written to data/splits/debug_20.txt "
    f"(first {N_DEBUG} IDs of train_10.txt), DataLoader batches of 4 confirmed working via a "
    f"custom collate_fn (dataset.py's KvasirSample dataclass isn't batchable by the default "
    f"collate — flagged for Sebastian, worked around locally, dataset.py untouched)."
)

Wrote 20 debug IDs to data/splits/debug_20.txt
Batch image shape: (4, 3, 256, 256)  dtype=torch.float32
Batch mask shape:  (4, 256, 256)  dtype=torch.float32
Batch image_ids:   ['cju5u6wf0kh1t0755bg1ssixv', 'cju5von04litr08718j8po40a', 'cju2rzpsmtb0f0835jabkbao1', 'cju5wphwwlu3m0987hh3ltg88']
PASS: DataLoader returns correctly-shaped batches, no errors.


## Task 5.4 — Set up loss and optimizer

In [6]:
def dice_loss(logits: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Soft Dice loss on sigmoid probabilities. Unlike metrics.py's dice() (a hard-mask
    EVALUATION metric, which the Week 1 Task 4 fix correctly stripped of EPS since its
    denominator is already zero-guarded), this EPS is legitimate: it's a small constant
    inside a differentiable TRAINING loss over continuous probabilities, not a metric being
    asserted against an exact value — different function, different purpose."""
    probs = torch.sigmoid(logits)
    probs_flat = probs.flatten(1)
    target_flat = target.flatten(1)
    intersection = (probs_flat * target_flat).sum(dim=1)
    denom = probs_flat.sum(dim=1) + target_flat.sum(dim=1)
    dice = (2 * intersection + eps) / (denom + eps)
    return 1.0 - dice.mean()


bce_loss = nn.BCEWithLogitsLoss()


def combined_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """configs/unet_base.yaml training.loss: dice_bce"""
    return bce_loss(logits, target) + dice_loss(logits, target)


optimizer = torch.optim.Adam(model.parameters(), lr=UNET_CFG["training"]["lr"])

print(f"Loss: BCEWithLogits + soft Dice (configs/unet_base.yaml training.loss=dice_bce)")
print(f"Optimizer: Adam, lr={UNET_CFG['training']['lr']} (configs/unet_base.yaml training.lr)")
results_log["5.4"] = (
    f"PASS — combined BCE + soft-Dice loss, Adam optimizer at lr={UNET_CFG['training']['lr']}, "
    f"both matching configs/unet_base.yaml (training.loss=dice_bce, training.optimizer=adam)."
)

Loss: BCEWithLogits + soft Dice (configs/unet_base.yaml training.loss=dice_bce)
Optimizer: Adam, lr=0.001 (configs/unet_base.yaml training.lr)


## Task 5.5 — Run the training loop for 2 epochs

Just proves it runs start to finish on the 20-image debug subset — not a real training run
(no validation split, no early stopping, no augmentation; those are Week 2).

In [7]:
N_EPOCHS = 2
epoch_losses = []
epoch_times = []

model.train()
t_start_total = time.perf_counter()

for epoch in range(1, N_EPOCHS + 1):
    t_start_epoch = time.perf_counter()
    running_loss = 0.0
    n_batches = 0

    for batch in debug_loader:
        images = batch["image"].to(DEVICE)
        targets = batch["mask"].unsqueeze(1).to(DEVICE)  # (B, H, W) -> (B, 1, H, W)

        optimizer.zero_grad()
        logits = model(images)
        loss = combined_loss(logits, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        n_batches += 1

    epoch_time = time.perf_counter() - t_start_epoch
    mean_loss = running_loss / n_batches
    epoch_losses.append(mean_loss)
    epoch_times.append(epoch_time)
    print(f"Epoch {epoch}/{N_EPOCHS}  mean loss={mean_loss:.4f}  time={epoch_time:.1f}s")

total_train_time = time.perf_counter() - t_start_total
print(f"\nCompleted {N_EPOCHS} epochs with no runtime errors (no shape mismatches, no NaNs).")
assert not any(np.isnan(epoch_losses)), "Loss went NaN — see notes/blockers before trusting anything downstream."
results_log["5.5"] = (
    f"PASS — {N_EPOCHS} epochs over {len(debug_dataset)} images completed with no runtime errors, "
    f"losses={[round(l, 4) for l in epoch_losses]}, total wall-clock={total_train_time:.1f}s."
)

Epoch 1/2  mean loss=1.3389  time=4.5s
Epoch 2/2  mean loss=1.2746  time=1.0s

Completed 2 epochs with no runtime errors (no shape mismatches, no NaNs).


## Task 5.6 — Confirm the loss actually decreases

In [8]:
print(f"Epoch 1 loss: {epoch_losses[0]:.4f}")
print(f"Epoch 2 loss: {epoch_losses[1]:.4f}")

if epoch_losses[1] < epoch_losses[0]:
    print("PASS: loss decreased between epoch 1 and epoch 2 — optimizer is updating weights.")
    results_log["5.6"] = f"PASS — loss dropped {epoch_losses[0]:.4f} -> {epoch_losses[1]:.4f} across the 2 epochs."
else:
    print("[WARN] Loss did NOT decrease — this is a plumbing test, so a flat/rising loss over just "
          "2 epochs on 20 images isn't necessarily fatal, but it's worth a second look before Week 2.")
    results_log["5.6"] = (
        f"WARN — loss did not decrease ({epoch_losses[0]:.4f} -> {epoch_losses[1]:.4f}). "
        f"Investigate before building Week 2's real training loop on top of this."
    )

Epoch 1 loss: 1.3389
Epoch 2 loss: 1.2746
PASS: loss decreased between epoch 1 and epoch 2 — optimizer is updating weights.


## Task 5.7 — Save a checkpoint and reload it

Saved under `checkpoints/` (already in `.gitignore`, so this won't get committed — it's a
throwaway smoke-test checkpoint, not one of the real Week 3 run checkpoints).

In [9]:
checkpoint_path = CHECKPOINTS_DIR / "unet_smoke_test.pt"
torch.save(model.state_dict(), checkpoint_path)
print(f"Saved checkpoint to {checkpoint_path.relative_to(REPO_ROOT)}")

# Reload into a FRESH model instance (not the one still in memory) to actually prove
# the checkpoint is self-sufficient, not just that the file was written.
reloaded_model = UNet(
    in_channels=3,
    out_channels=1,
    base_channels=UNET_CFG["model"]["base_channels"],
    depth=UNET_CFG["model"]["depth"],
).to(DEVICE)
reloaded_model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
reloaded_model.eval()

with torch.no_grad():
    reload_batch = next(iter(debug_loader))
    reload_out = reloaded_model(reload_batch["image"].to(DEVICE))

print(f"Reloaded model forward pass output shape: {tuple(reload_out.shape)}")
print("PASS: checkpoint saved and reloaded into a fresh model instance with no errors.")
results_log["5.7"] = (
    f"PASS — checkpoint saved to checkpoints/unet_smoke_test.pt and reloaded into a fresh "
    f"UNet instance, forward pass on a real batch ran with no errors."
)

Saved checkpoint to checkpoints/unet_smoke_test.pt
Reloaded model forward pass output shape: (4, 1, 256, 256)
PASS: checkpoint saved and reloaded into a fresh model instance with no errors.


## Task 5.8 — Log runtime, GPU memory, and any bugs hit

Feeds the Week 4 compute/amortisation table later — capturing it now while it's live.

In [10]:
if DEVICE.type == "cuda":
    peak_mem_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2)
    mem_note = f"{peak_mem_mb:.1f} MB peak CUDA memory (torch.cuda.max_memory_allocated)"
elif DEVICE.type == "mps":
    # torch.mps memory accounting (added in torch 2.0) — best-effort, not as exact as CUDA's.
    try:
        peak_mem_mb = torch.mps.driver_allocated_memory() / (1024 ** 2)
        mem_note = f"{peak_mem_mb:.1f} MB MPS driver-allocated memory (approximate, no per-run peak reset on MPS)"
    except AttributeError:
        mem_note = "MPS memory accounting unavailable in this torch version"
else:
    mem_note = "No GPU memory metric on CPU"

# Fill this in by hand with anything you actually hit while running the cells above —
# this list is deliberately empty by default rather than guessing bugs that didn't happen.
bugs_hit = [
    # e.g. "shape mismatch on first run because resize wasn't applied before transform — fixed by X",
]

print(f"Device: {DEVICE} ({device_name})")
print(f"Total training wall-clock ({N_EPOCHS} epochs, {len(debug_dataset)} images): {total_train_time:.1f}s")
print(f"Per-epoch times: {[round(t, 1) for t in epoch_times]}")
print(f"Memory: {mem_note}")
print(f"Bugs hit: {bugs_hit if bugs_hit else 'none'}")

bugs_note = "; ".join(bugs_hit) if bugs_hit else "none"
results_log["5.8"] = (
    f"Device={DEVICE} ({device_name}), total wall-clock={total_train_time:.1f}s over {N_EPOCHS} "
    f"epochs / {len(debug_dataset)} images, {mem_note}. Bugs hit: {bugs_note}."
)

Device: mps (Apple Silicon (mps))
Total training wall-clock (2 epochs, 20 images): 5.5s
Per-epoch times: [4.5, 1.0]
Memory: 2165.9 MB MPS driver-allocated memory (approximate, no per-run peak reset on MPS)
Bugs hit: none


## Append entry to `decisions.md`

Builds one dated entry from the results of 5.1–5.8 above and appends it to `decisions.md`,
matching the same `Date | Decision | Rationale | Decided by` table format used for Task 2.7.

**Before running this cell:** `decisions.md` currently has unresolved git merge-conflict
markers committed into it from the PR #1 merge (`<<<<<<<`/`=======`/`>>>>>>>` lines near the
bottom) — both sides' content looks legitimate, it just needs those 3 marker lines deleted.
Fix that first, or this cell's append will land after the broken markers rather than at a
clean end of file.

In [11]:
today = date.today().isoformat()

entry_lines = [
    f"| {today} | Week 1 Task 5 — U-Net smoke test | {results_log.get('5.1', 'not run')} | Balamurali |",
    f"| {today} | 5.2 U-Net architecture | {results_log.get('5.2', 'not run')} | Balamurali |",
    f"| {today} | 5.3 debug-subset wiring | {results_log.get('5.3', 'not run')} | Balamurali |",
    f"| {today} | 5.4 loss + optimizer | {results_log.get('5.4', 'not run')} | Balamurali |",
    f"| {today} | 5.5 training loop (2 epochs) | {results_log.get('5.5', 'not run')} | Balamurali |",
    f"| {today} | 5.6 loss decreased | {results_log.get('5.6', 'not run')} | Balamurali |",
    f"| {today} | 5.7 checkpoint save/reload | {results_log.get('5.7', 'not run')} | Balamurali |",
    f"| {today} | 5.8 runtime/memory/bugs | {results_log.get('5.8', 'not run')} | Balamurali |",
]

header = "# Decisions Log\n\n| Date | Decision | Rationale | Decided by |\n|---|---|---|---|\n"

if not DECISIONS_MD.exists():
    DECISIONS_MD.write_text(header + "\n".join(entry_lines) + "\n")
    print(f"Created {DECISIONS_MD} and added the entry.")
else:
    with open(DECISIONS_MD, "a") as f:
        f.write("\n".join(entry_lines) + "\n")
    print(f"Appended the entry to {DECISIONS_MD}.")

print("\n--- entry written ---")
print("\n".join(entry_lines))

Appended the entry to /Volumes/Crucial X9/home/Developer/COMPSCI-764-research/decisions.md.

--- entry written ---
| 2026-09-25 | Week 1 Task 5 — U-Net smoke test | WARN — no CUDA device, ran on mps (Apple Silicon (mps)), torch 2.11.0. requirements.txt versions are not yet pinned (open TODO in the file itself). | Balamurali |
| 2026-09-25 | 5.2 U-Net architecture | PASS — UNet(depth=4, base_channels=32) in src/unet.py, 7,763,041 params, output shape torch.Size([2, 1, 256, 256]) matches mask shape, no pretrained weights. | Balamurali |
| 2026-09-25 | 5.3 debug-subset wiring | PASS — 20-image debug split written to data/splits/debug_20.txt (first 20 IDs of train_10.txt), DataLoader batches of 4 confirmed working via a custom collate_fn (dataset.py's KvasirSample dataclass isn't batchable by the default collate — flagged for Sebastian, worked around locally, dataset.py untouched). | Balamurali |
| 2026-09-25 | 5.4 loss + optimizer | PASS — combined BCE + soft-Dice loss, Adam optimizer at 